**Gas Mixing: Mixing Three Light Hydrocarbon Streams**

In this tutorial we mix three pure gas streams (methane, ethane and propane) in an adiabatic mixer using the Peng-Robinson equation of state, and validate the result against an equivalent DWSIM model (Peng-Robinson property package).

We assume that the mixer operates at steady state with no heat loss.

### Inlet Specifications:

| Stream | Component | Mass Flow | Temperature | Pressure |
|---|---|---|---|---|
| 1 | Methane | 3600 kg/h | 80 °C | 1.01325 bar |
| 2 | Ethane | 3600 kg/h | 50 °C | 1.01325 bar |
| 3 | Propane | 3600 kg/h | 25 °C | 1.01325 bar |

### Mixer Specifications:

* **Outlet pressure**: minimum of the inlet pressures (`MomentumMixingType.minimize`, DWSIM "Minimum")
* **Energy balance**: adiabatic, so the outlet temperature follows from the enthalpy balance

---

> **Matching the DWSIM thermodynamics**
>
> The property package `light_hydrocarbons.py` uses the same Peng-Robinson binary interaction parameters (kij) as DWSIM. Ideal gas heat capacities come from the Poling correlations in the `chemicals` library, which differ slightly from DWSIM's database, so a small outlet temperature difference is expected.

In [ ]:
import idaes

print("IDAES version: ", idaes.__version__)

## Setting up and solving the flowsheet

The flowsheet is three Feeds → Mixer → Product, built in `mixer.py`. Degrees of freedom are printed before and after fixing the feed conditions.

In [ ]:
from mixer import setup_mixer_model, report_mixer_properties

# Create and solve the model
model, results = setup_mixer_model(
    streams={
        "methane": {"flow_mass": 1.0, "temperature": 80.0, "pressure": 101325},  # kg/s, C, Pa
        "ethane": {"flow_mass": 1.0, "temperature": 50.0, "pressure": 101325},
        "propane": {"flow_mass": 1.0, "temperature": 25.0, "pressure": 101325},
    }
)
print(results.solver.termination_condition)

prop = report_mixer_properties(model)

In [ ]:
# Visualize (if needed)
model.fs.visualize("Mixer Model")

## DWSIM model

In [ ]:
from IPython.display import Image
Image(filename="DWSim Mixer.png")

In [ ]:
import webbrowser
import tempfile
import os

def display_image_in_browser(image_path, title="Image Viewer"):
    # Create a temporary HTML file that embeds the image
    html_content = f"""
    <html>
        <head>
            <title>{title}</title>
        </head>
        <body style="text-align: center;">
            <h2>{title}</h2>
            <img src="file://{os.path.abspath(image_path)}" style="max-width:90%; max-height:90%;">
        </body>
    </html>
    """
    with tempfile.NamedTemporaryFile('w', delete=False, suffix='.html') as f:
        f.write(html_content)
        temp_html_path = f.name

    webbrowser.open(f'file://{temp_html_path}')


In [ ]:
display_image_in_browser("DWSim Mixer.png", title="DWSim Mixer Model Check")

## Validation against DWSIM

The DWSIM results are read directly from the solved `.dwxmz` file and compared with IDAES. Flows and composition follow from the mass balance and should match exactly; the outlet temperature differs slightly because of the ideal gas heat capacity data.

In [ ]:
from mixer import read_dwsim_results, compare_with_dwsim

dwsim = read_dwsim_results("DWSim Mixer Model.dwxmz")
compare_with_dwsim(model, dwsim)